<a href="https://colab.research.google.com/github/9854945d/llm-research-toolkit/blob/main/hydrologic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
### 1. Google Drive 연결 (Mount)
계정 권한을 승인하여 드라이브를 연결합니다.
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


```markdown
### 2. HydroSHEDS 관련 .tif 파일 검색
파일명에 'Hydro', 'SHEDS', 또는 'dem', 'acc', 'dir' 등이 포함된 `.tif` 파일을 탐색합니다.
```

In [ ]:
!pip install pysheds
!pip install rasterio
!pip install geopandas
!pip install shapely
!pip install matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
from pysheds.grid import Grid
import geopandas as gpd
from shapely.geometry import Polygon
import rasterio

# Google Drive 마운트 (이미 되어있다면 생략 가능)
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# [설정 1] DEM 파일 경로 지정
# ==========================================
# 각하의 실제 파일 경로로 수정해 주십시오.
dem_path = '/content/drive/MyDrive/Hydrology lab/GeoData/Dem_Korea_Clip.tif'

# ==========================================
# [설정 2] 추출할 대권역 하구(출구점) 좌표 설정 (경도, 위도)
# ==========================================
# 예시: 한강 하구 (강화도 인근), 낙동강 하구둑 부근의 대략적 좌표
# 원하는 대권역 하구 좌표를 WGS84(경도, 위도) 기준으로 입력합니다.
target_basins = {
    'Han_River': (126.55, 37.75),    # 한강 하구 (대략적)
    'Nakdong_River': (128.93, 35.10) # 낙동강 하구둑 (대략적)
}

print("1. DEM 로딩 및 지형 평탄화 작업 시작 (시간이 약간 소요됩니다)...")
# Grid 인스턴스 생성 및 DEM 로딩
grid = Grid.from_raster(dem_path)
dem = grid.read_raster(dem_path)

# DEM 노이즈 제거 (웅덩이(Pit) 및 함몰지(Depression) 채우기)
pit_filled_dem = grid.fill_pits(dem)
flooded_dem = grid.fill_depressions(pit_filled_dem)

# 흐름 방향(Flow Direction) 및 누적 유량(Flow Accumulation) 계산
# dirmap: 지형을 따라 물이 흐르는 8가지 방향
dirmap = (64, 128, 1, 2, 4, 8, 16, 32)
fdir = grid.flowdir(flooded_dem, dirmap=dirmap)
acc = grid.accumulation(fdir, dirmap=dirmap)
print("-> 지형 흐름 연산 완료.")

# 결과를 저장할 리스트
basin_polygons = []

print("\n2. 대권역 유역 추출 시작...")
for basin_name, (lon, lat) in target_basins.items():
    print(f"[{basin_name}] 하구 좌표({lon}, {lat}) 기반 유역 추적 중...")

    # ---------------------------------------------------------
    # [핵심] Snap Pour Point: 입력한 좌표를 물이 가장 많이 모이는 '진짜 하천'으로 강제 이동
    # ---------------------------------------------------------
    # snap_distance: 하구 좌표가 약간 틀려도 주변 하천을 찾을 반경
    x_snap, y_snap = grid.snap_to_mask(acc > 10000, (lon, lat))

    # 유역 경계 계산 (Catchment Delineation)
    catch = grid.catchment(x=x_snap, y=y_snap, fdir=fdir, dirmap=dirmap,
                           xytype='coordinate')

    # 그리드 결과를 벡터 폴리곤으로 변환
    shapes = grid.polygonize(catch)

    # 유효한 폴리곤이 여러 개 나올 경우 가장 큰 것(본류) 하나만 선택
    max_poly = None
    max_area = 0
    for shape, value in shapes:
        # value == 1 인 영역이 실제 유역 내부
        if value == 1:
            poly = Polygon(shape['coordinates'][0])
            if poly.area > max_area:
                max_area = poly.area
                max_poly = poly

    if max_poly:
        basin_polygons.append({
            'Basin_Name': basin_name,
            'geometry': max_poly
        })
        print(f"-> {basin_name} 유역 폴리곤 생성 완료.")
    else:
        print(f"-> [오류] {basin_name} 유역 생성 실패. 하구 좌표를 수정해 보세요.")

# ==========================================
# 3. 추출된 유역을 Shapefile로 저장 및 시각화
# ==========================================
if basin_polygons:
    # GeoDataFrame으로 변환
    gdf_basins = gpd.GeoDataFrame(basin_polygons, crs=grid.crs)

    # Shapefile 저장 경로 설정 (원하시는 경로로 변경 가능)
    save_dir = '/content/drive/MyDrive/Hydrology lab/GeoData/Korea_watershed_shp/Macro_Basins'
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, 'Macro_Basins_Extracted.shp')

    # Shapefile 내보내기
    gdf_basins.to_file(save_path)
    print(f"\n3. 완료! 대권역 Shapefile이 저장되었습니다: {save_path}")

    # 시각적 확인 (Plot)
    fig, ax = plt.subplots(figsize=(8, 10))
    # DEM 배경 흐리게 표시
    im = ax.imshow(dem, extent=grid.extent, cmap='terrain', alpha=0.5, vmin=0, vmax=1500)
    # 추출한 대권역 외곽선 표시
    gdf_basins.boundary.plot(ax=ax, color='red', linewidth=2)
    plt.title("Extracted Macro Basins (Red Line)", fontsize=16)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()
else:
    print("생성된 유역이 없습니다.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.8 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. DEM 로딩 및 지형 평탄화 작업 시작 (시간이 약간 소요됩니다)...


RasterioIOError: /content/drive/MyDrive/Hydrology lab/GeoData/Dem_Korea_Clip.tif: No such file or directory